# R2 (research track): chunk-level labels for rendering-aware routing

Second milestone of the README's Phase 4 ([proposal](../docs/research/hybridsfm_proposal.md)). R1 showed that standard pose metrics miss what matters for the twin: VGGT scored AUC@30 = 0.995 on garden and still lost 4.2 dB. Which reconstructor wins also changes with the scene. R2 builds the dataset the router learns from.

**For every chunk** (an image subset of one scene: 15 per scene, 12–40 images, consecutive windows, wide-baseline strides and random sets):

| Method | What it is | Runs on |
|---|---|---|
| `ref` | Official Mip-NeRF 360 poses (COLMAP on all images), cut down to the chunk. The reference | – |
| `classical` | ALIKED + LightGlue matches → COLMAP verification + incremental mapper (hloc-style; SIFT fallback) | GPU + CPU |
| `vggt` | VGGT-1B, one forward pass (`demo_colmap.py`). Its raw predictions are saved for R3's signals | GPU |
| `vggt_ba` | VGGT + bundle adjustment (lightest tracking settings, model moved off the GPU: F-R1.3) | GPU + CPU |

Each method's poses are scored against `ref` with tighter metrics than R1 (AUC@3/@5 as well as @30, focal error). Then the **same short 3DGS recipe** is trained on each (MCMC 300k cap, 3k steps, every 8th image held out). The label is the PSNR loss against `ref`.

**On 2 chunks per scene, two extra checks run:**
- **Fidelity:** a 7k-step run, to check that 3k-step labels rank methods the same way (Spearman ρ).
- **Initialisation control:** each method also trained from the *reference* point cloud moved into its frame. This separates pose error from the different starting points seen in R1.

**Outputs** (all small):
- `r2/r2_labels.jsonl`: one line per run; the dataset.
- `r2/r2_summary.md`: routing baselines, oracle, the pose-metric vs twin-loss check, fidelity, initialisation control. Paste it back into the chat.
- `r2/r2_pred/`: VGGT's raw predictions per chunk (for R3).
- `r2/r2_models/`: every method's COLMAP model per chunk.
- `r2/r2_logs/`: logs of failed runs.

**Resolution:** outdoor scenes use `images_8` (~620 px wide), indoor scenes `images_4` (~780 px). Labels compare methods on the same images, so absolute PSNR matters less than the gaps.

**How to run (Colab):**
1. `Runtime → Change runtime type → T4 GPU` → **Run all**. If the runtime restarts once after installing (numpy), press **Run all** again.
2. Keep the tab open. Each chunk is about 10 min; the full set (~105 chunks) is about **20 GPU-hours**, so this takes several sessions.
3. **When Colab disconnects or hits its GPU limit:** press **Run all** in the next session. Every finished run is in `3dgs-digital-twin/r2/r2_labels.jsonl` on Drive and is skipped. The installs repeat (~10 min; the gsplat build comes from the Drive cache).

You can paste `r2/r2_summary.md` after any session. It covers everything labelled so far.

## 1. Settings

In [ ]:
import os, sys, glob, json, time, re, subprocess, shutil
from google.colab import drive
drive.mount('/content/drive')
print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total,driver_version', '--format=csv'],
                     capture_output=True, text=True).stdout)
BUDGET_DEFAULT = 24.0                        # Colab ends sessions itself; everything resumes from Drive

ALL_SCENES = ['garden', 'bicycle', 'stump', 'bonsai', 'counter', 'kitchen', 'room']   # Mip-NeRF 360 (public zip)
SCENES     = ALL_SCENES                       # split across runs if needed, e.g. ALL_SCENES[:4]
RES        = dict(garden='images_8', bicycle='images_8', stump='images_8',            # ~620 px wide
                  bonsai='images_4', counter='images_4', kitchen='images_4', room='images_4')   # ~780 px wide
METHODS    = ['ref', 'classical', 'vggt', 'vggt_ba']
STEPS, LONG_STEPS, CAP = 3_000, 7_000, 300_000  # label recipe; fidelity check; Gaussian cap
FIDELITY   = ['window24_0', 'stride24_0']       # per scene: also trained at LONG_STEPS and from the reference points
MIN_COVERAGE = 0.9                              # a method must register >= 90% of the chunk to get a 3DGS label
BUDGET_H   = BUDGET_DEFAULT                     # stop starting new chunks after this many hours (summary still written)
T_START    = time.time()

PROJECT = '/content/drive/MyDrive/3dgs-digital-twin'   # Drive: survives disconnects
WORK    = '/content/work'
OUT     = f'{PROJECT}/r2'
R2      = f'{WORK}/r2'                       # per-chunk, per-method scene folders (images are symlinks)
LABELS  = f'{OUT}/r2_labels.jsonl'
for d in [f'{PROJECT}/cache', OUT, f'{OUT}/r2_pred', f'{OUT}/r2_models', f'{OUT}/r2_logs', WORK, f'{WORK}/hybrid_sfm', R2]:
    os.makedirs(d, exist_ok=True)

## 2. Install gsplat (main environment)

Same cells as R1 (compiled library reused from the Drive cache).

In [ ]:
import importlib.util
MARKER = '/content/.gs_deps_ok'          # /content survives a kernel restart (not a VM reset)

if not os.path.exists(MARKER):
    import torch
    np_before = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                               capture_output=True, text=True).stdout.strip()
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'gsplat==1.5.3'], check=True)

    REPO = f'{WORK}/gsplat'
    if not os.path.exists(REPO):
        subprocess.run(['git', 'clone', '-q', '--depth', '1', '--branch', 'v1.5.3',
                        'https://github.com/nerfstudio-project/gsplat.git', REPO], check=True)
    reqs = open(f'{REPO}/examples/requirements.txt').read()

    # F0.1 + F0.2: git deps (several glued onto one line) -> split, build without isolation
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'ninja', 'setuptools', 'wheel'], check=True)
    if 'T4' in torch.cuda.get_device_name(0):
        os.environ['TORCH_CUDA_ARCH_LIST'] = '7.5'   # faster builds, T4 only
    for url in re.findall(r'git\+https://github\.com/[^@\s]+@[0-9a-f]{40}', reqs):
        print('building', url.split('/')[-1][:60])
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '--no-build-isolation', url], check=True)

    # F0.3: drop git lines + numpy<2 pin, install the rest
    rest = [l for l in reqs.splitlines() if 'git+https' not in l and not l.lower().startswith('numpy')]
    open('/tmp/reqs.txt', 'w').write('\n'.join(rest))
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r', '/tmp/reqs.txt'], check=True)
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'remotezip', 'plyfile', 'numpy>=2.1,<2.5'], check=True)

    # F0.5: pycolmap fork uses np.uint64(-1) -> OverflowError on numpy 2 (patch the file, don't import it)
    f = os.path.join(os.path.dirname(importlib.util.find_spec('pycolmap').origin), 'scene_manager.py')
    s = open(f).read()
    if 'np.uint64(-1)' in s:
        open(f, 'w').write(s.replace('np.uint64(-1)', 'np.uint64(18446744073709551615)')); print('patched pycolmap')

    # F0.4: make gsplat's examples/datasets a regular package (beats HF `datasets`)
    open(f'{REPO}/examples/datasets/__init__.py', 'a').close()

    open(MARKER, 'w').write('ok')
    np_after = subprocess.run([sys.executable, '-c', 'import numpy; print(numpy.__version__)'],
                              capture_output=True, text=True).stdout.strip()
    if np_before != np_after:
        print(f'numpy {np_before} -> {np_after}: restarting runtime. Then "Run all" again.')
        time.sleep(2); os.kill(os.getpid(), 9)

REPO = f'{WORK}/gsplat'
check = subprocess.run([sys.executable, '-c',
    'import numpy, gsplat, fused_ssim, pycolmap, tyro; print("numpy", numpy.__version__, "| gsplat", gsplat.__version__)'],
    capture_output=True, text=True, cwd=f'{REPO}/examples')
print(check.stdout or check.stderr[-2000:])

In [ ]:
# F1.5 (second fix): keep gsplat's compiled CUDA library on Drive and load it directly in later sessions
import torch, importlib.util
PREBUILT = '/content/gsplat_prebuilt'
cap = torch.cuda.get_device_capability()
KEY = f"gsplat1.5.3_torch{torch.__version__}_py{sys.version_info.major}{sys.version_info.minor}_sm{cap[0]}{cap[1]}"
JIT_CACHE = f'{PROJECT}/cache/gsplat_jit/{KEY}'

# 1. Patch gsplat's loader: use /content/gsplat_prebuilt/gsplat_cuda.so if it exists, otherwise behave as before
BACKEND = os.path.join(os.path.dirname(importlib.util.find_spec('gsplat').origin), 'cuda', '_backend.py')
src = open(BACKEND).read()
if 'GSPLAT_PREBUILT_DIR' not in src:
    old = '    from gsplat import csrc as _C\n'
    new = ('    _pre = os.environ.get("GSPLAT_PREBUILT_DIR", "/content/gsplat_prebuilt")\n'
           '    if os.path.exists(os.path.join(_pre, "gsplat_cuda.so")):\n'
           '        _C = _import_module_from_library("gsplat_cuda", _pre, True)\n'
           '    else:\n'
           '        from gsplat import csrc as _C\n')
    assert src.count(old) == 1, 'gsplat _backend.py changed, patch needs updating'
    open(BACKEND, 'w').write(src.replace(old, new)); print('patched gsplat loader')

# 2. Restore the cached library from Drive
os.makedirs(PREBUILT, exist_ok=True)
if not os.path.exists(f'{PREBUILT}/gsplat_cuda.so') and os.path.exists(f'{JIT_CACHE}/gsplat_cuda.so'):
    shutil.copy(f'{JIT_CACHE}/gsplat_cuda.so', PREBUILT)

def load_gsplat_cuda():
    t = time.time()
    r = subprocess.run([sys.executable, '-c', 'from gsplat.cuda._backend import _C; print("SO=", _C.__file__ if _C else None)'],
                       capture_output=True, text=True)
    so = re.search(r'SO= (.+)', r.stdout)
    return (so.group(1).strip() if so else None), time.time() - t, r

so, dt, r = load_gsplat_cuda()
if so and so.startswith(PREBUILT):
    print(f'✅ gsplat CUDA loaded from the Drive cache in {dt:.0f}s ({KEY})')
else:
    # 3. No usable cache: the import above compiled it (~15 min). Save the result for next time.
    if not so or so == 'None':
        print(r.stdout[-1500:], r.stderr[-3000:]); raise RuntimeError('gsplat CUDA failed to load or compile')
    print(f'compiled gsplat CUDA in {dt/60:.1f} min -> saving to Drive')
    os.makedirs(JIT_CACHE, exist_ok=True)
    shutil.copy(so, f'{JIT_CACHE}/gsplat_cuda.so'); shutil.copy(so, f'{PREBUILT}/gsplat_cuda.so')
    so2, dt2, _ = load_gsplat_cuda()
    print(f"{'✅' if so2 and so2.startswith(PREBUILT) else '❌'} reload from cache: {dt2:.0f}s ({so2})")

## 3. VGGT environment (separate Python 3.12, official pycolmap 3.10, LightGlue)

Same as R1. The classical reconstructor runs here too.

In [ ]:
VENV, VGGT_REPO = f'{WORK}/vggt_env', f'{WORK}/vggt'
VPY = f'{VENV}/bin/python'
def sh(cmd, **kw):
    r = subprocess.run(cmd, capture_output=True, text=True, **kw)
    if r.returncode != 0:
        print(r.stdout[-3000:], r.stderr[-3000:]); raise RuntimeError(f'failed: {" ".join(map(str, cmd))[:200]}')
    return r.stdout

if not os.path.exists(f'{VENV}/.ok'):
    t = time.time()
    sh([sys.executable, '-m', 'pip', 'install', '-q', 'uv'])
    sh([sys.executable, '-m', 'uv', 'venv', '--python', '3.12', VENV])
    UV = [sys.executable, '-m', 'uv', 'pip', 'install', '--python', VPY]
    sh(UV + ['torch==2.3.1', 'torchvision==0.18.1', '--extra-index-url', 'https://download.pytorch.org/whl/cu121'])
    sh(UV + ['numpy==1.26.4', 'Pillow', 'huggingface_hub', 'einops', 'safetensors', 'opencv-python-headless',
             'trimesh', 'scipy', 'tqdm', 'pycolmap==3.10.0', 'pyceres==2.3',
             'lightglue @ git+https://github.com/jytime/LightGlue.git'])
    if not os.path.exists(VGGT_REPO):
        sh(['git', 'clone', '-q', '--depth', '1', 'https://github.com/facebookresearch/vggt.git', VGGT_REPO])
    open(f'{VENV}/.ok', 'w').write('ok')
    print(f'VGGT environment ready in {(time.time() - t) / 60:.1f} min')

VGGT_COMMIT = sh(['git', '-C', VGGT_REPO, 'rev-parse', '--short', 'HEAD']).strip()
VENV_ENV = {**os.environ, 'PYTHONPATH': VGGT_REPO, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
if not os.path.exists(f'{VENV}/.demo_ok'):            # everything demo_colmap.py imports, minus the web-demo extras
    reqs = [l.strip() for f in ['requirements.txt', 'requirements_demo.txt'] if os.path.exists(f'{VGGT_REPO}/{f}')
            for l in open(f'{VGGT_REPO}/{f}') if l.strip() and not l.startswith('#')]
    reqs = [r for r in reqs if not re.match(r'(gradio|viser|onnxruntime|torch|torchvision|numpy|pycolmap|pyceres|lightglue)\b', r, re.I)]
    sh([sys.executable, '-m', 'uv', 'pip', 'install', '--python', VPY, *reqs,
        'numpy==1.26.4', 'pycolmap==3.10.0', 'pyceres==2.3', 'matplotlib'])
    open(f'{VENV}/.demo_ok', 'w').write('ok')
r = subprocess.run([VPY, '-c', 'import demo_colmap'], cwd=VGGT_REPO, env=VENV_ENV, capture_output=True, text=True)
if r.returncode != 0:
    print(r.stderr[-3000:]); raise RuntimeError('demo_colmap.py cannot be imported: see the error above')
print(sh([VPY, '-c', 'import torch, pycolmap, pyceres, lightglue; '
          'print("torch", torch.__version__, "| cuda", torch.cuda.is_available(), "| pycolmap", pycolmap.__version__)'],
         env=VENV_ENV).strip(), '| vggt', VGGT_COMMIT)

## 4. Research code

Copies of `hybrid_sfm/` in the repo, tested locally (`python hybrid_sfm/tests/test_r1.py`, `test_r2.py`).

In [ ]:
%%writefile /content/work/hybrid_sfm/colmap_io.py
"""Minimal COLMAP model I/O (binary and text), numpy only.

Written for the research track so pose evaluation and subsetting don't depend on a particular pycolmap
version (gsplat's examples install a pycolmap *fork* that shadows the official package; VGGT's
demo_colmap.py needs official pycolmap 3.10).

Model = dict with
    cameras: {camera_id: dict(model, width, height, params)}
    images:  {image_id: dict(qvec (w,x,y,z), tvec, camera_id, name, xys (P,2), point3D_ids (P,))}
    points:  {point3D_id: dict(xyz, rgb, error, image_ids, point2D_idxs)}
Poses are world-to-camera (COLMAP / OpenCV convention): x_cam = R(q) @ x_world + t.
"""
import os
import struct

import numpy as np

CAMERA_MODELS = {  # id: (name, number of params)
    0: ("SIMPLE_PINHOLE", 3), 1: ("PINHOLE", 4), 2: ("SIMPLE_RADIAL", 4), 3: ("RADIAL", 5),
    4: ("OPENCV", 8), 5: ("OPENCV_FISHEYE", 8), 6: ("FULL_OPENCV", 12), 7: ("FOV", 5),
    8: ("SIMPLE_RADIAL_FISHEYE", 4), 9: ("RADIAL_FISHEYE", 5), 10: ("THIN_PRISM_FISHEYE", 12),
}
MODEL_ID = {name: i for i, (name, _) in CAMERA_MODELS.items()}


def _read(f, fmt):
    return struct.unpack("<" + fmt, f.read(struct.calcsize("<" + fmt)))


# ------------------------------------------------------------------ binary
def read_cameras_bin(path):
    cams = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            cid, mid, w, h = _read(f, "iiQQ")
            name, n = CAMERA_MODELS[mid]
            cams[cid] = dict(model=name, width=w, height=h, params=np.array(_read(f, "d" * n)))
    return cams


def read_images_bin(path):
    imgs = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            iid, = _read(f, "i")
            q = np.array(_read(f, "dddd"))
            t = np.array(_read(f, "ddd"))
            cid, = _read(f, "i")
            name = b""
            while (c := f.read(1)) != b"\x00":
                name += c
            n, = _read(f, "Q")
            raw = np.frombuffer(f.read(24 * n), dtype=np.dtype([("x", "<f8"), ("y", "<f8"), ("id", "<i8")]))
            imgs[iid] = dict(qvec=q, tvec=t, camera_id=cid, name=name.decode(),
                             xys=np.stack([raw["x"], raw["y"]], 1), point3D_ids=raw["id"].copy())
    return imgs


def read_points_bin(path):
    pts = {}
    with open(path, "rb") as f:
        for _ in range(_read(f, "Q")[0]):
            pid, = _read(f, "Q")
            xyz = np.array(_read(f, "ddd"))
            rgb = np.array(_read(f, "BBB"), np.uint8)
            err, n = _read(f, "dQ")
            tr = np.frombuffer(f.read(8 * n), dtype="<i4").reshape(-1, 2)
            pts[pid] = dict(xyz=xyz, rgb=rgb, error=err, image_ids=tr[:, 0].copy(), point2D_idxs=tr[:, 1].copy())
    return pts


def write_model_bin(model, out_dir):
    os.makedirs(out_dir, exist_ok=True)
    with open(os.path.join(out_dir, "cameras.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["cameras"])))
        for cid, c in sorted(model["cameras"].items()):
            f.write(struct.pack("<iiQQ", cid, MODEL_ID[c["model"]], int(c["width"]), int(c["height"])))
            f.write(struct.pack("<" + "d" * len(c["params"]), *map(float, c["params"])))
    with open(os.path.join(out_dir, "images.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["images"])))
        for iid, im in sorted(model["images"].items()):
            f.write(struct.pack("<i4d3di", iid, *map(float, im["qvec"]), *map(float, im["tvec"]), im["camera_id"]))
            f.write(im["name"].encode() + b"\x00")
            xys, ids = np.asarray(im["xys"], float).reshape(-1, 2), np.asarray(im["point3D_ids"], np.int64)
            f.write(struct.pack("<Q", len(ids)))
            rec = np.empty(len(ids), dtype=np.dtype([("x", "<f8"), ("y", "<f8"), ("id", "<i8")]))
            rec["x"], rec["y"], rec["id"] = xys[:, 0], xys[:, 1], ids
            f.write(rec.tobytes())
    with open(os.path.join(out_dir, "points3D.bin"), "wb") as f:
        f.write(struct.pack("<Q", len(model["points"])))
        for pid, p in sorted(model["points"].items()):
            f.write(struct.pack("<Q3d3Bd", pid, *map(float, p["xyz"]), *map(int, p["rgb"]), float(p["error"])))
            tr = np.stack([p["image_ids"], p["point2D_idxs"]], 1).astype("<i4")
            f.write(struct.pack("<Q", len(tr)) + tr.tobytes())


# ------------------------------------------------------------------ text (read only)
def _data_lines(path):
    with open(path) as f:
        return [l.rstrip("\n") for l in f if not l.startswith("#")]


def read_model_txt(d):
    cams, imgs, pts = {}, {}, {}
    for l in _data_lines(os.path.join(d, "cameras.txt")):
        if not l.strip():
            continue
        v = l.split()
        cams[int(v[0])] = dict(model=v[1], width=int(v[2]), height=int(v[3]), params=np.array(v[4:], float))
    lines = _data_lines(os.path.join(d, "images.txt"))   # pairs: header line, observations line (may be empty)
    i = 0
    while i < len(lines):
        if not lines[i].strip():
            i += 1
            continue
        v = lines[i].split()
        obs = np.array(lines[i + 1].split(), float).reshape(-1, 3) if i + 1 < len(lines) else np.zeros((0, 3))
        imgs[int(v[0])] = dict(qvec=np.array(v[1:5], float), tvec=np.array(v[5:8], float), camera_id=int(v[8]),
                               name=" ".join(v[9:]), xys=obs[:, :2], point3D_ids=obs[:, 2].astype(np.int64))
        i += 2
    for l in _data_lines(os.path.join(d, "points3D.txt")):
        if not l.strip():
            continue
        v = l.split()
        tr = np.array(v[8:], int).reshape(-1, 2)
        pts[int(v[0])] = dict(xyz=np.array(v[1:4], float), rgb=np.array(v[4:7], np.uint8), error=float(v[7]),
                              image_ids=tr[:, 0], point2D_idxs=tr[:, 1])
    return dict(cameras=cams, images=imgs, points=pts)


def find_model_dir(scene_dir):
    """gsplat's rule: sparse/0 if it exists, else sparse."""
    for d in (os.path.join(scene_dir, "sparse", "0"), os.path.join(scene_dir, "sparse")):
        if os.path.exists(os.path.join(d, "images.bin")) or os.path.exists(os.path.join(d, "images.txt")):
            return d
    raise FileNotFoundError(f"no COLMAP model under {scene_dir}/sparse")


def read_model(d):
    if os.path.isdir(os.path.join(d, "sparse")):
        d = find_model_dir(d)
    if os.path.exists(os.path.join(d, "images.bin")):
        return dict(cameras=read_cameras_bin(os.path.join(d, "cameras.bin")),
                    images=read_images_bin(os.path.join(d, "images.bin")),
                    points=read_points_bin(os.path.join(d, "points3D.bin")))
    return read_model_txt(d)


# ------------------------------------------------------------------ helpers
def qvec2rotmat(q):
    w, x, y, z = q / np.linalg.norm(q)
    return np.array([[1 - 2 * (y * y + z * z), 2 * (x * y - w * z), 2 * (x * z + w * y)],
                     [2 * (x * y + w * z), 1 - 2 * (x * x + z * z), 2 * (y * z - w * x)],
                     [2 * (x * z - w * y), 2 * (y * z + w * x), 1 - 2 * (x * x + y * y)]])


def rotmat2qvec(R):
    K = np.array([[R[0, 0] - R[1, 1] - R[2, 2], 0, 0, 0],
                  [R[1, 0] + R[0, 1], R[1, 1] - R[0, 0] - R[2, 2], 0, 0],
                  [R[2, 0] + R[0, 2], R[2, 1] + R[1, 2], R[2, 2] - R[0, 0] - R[1, 1], 0],
                  [R[2, 1] - R[1, 2], R[0, 2] - R[2, 0], R[1, 0] - R[0, 1], R[0, 0] + R[1, 1] + R[2, 2]]]) / 3
    w, V = np.linalg.eigh(K)
    q = V[[3, 0, 1, 2], np.argmax(w)]
    return q if q[0] >= 0 else -q


def poses_by_name(model):
    """{image name: (R (3,3), t (3,))}, world-to-camera."""
    return {im["name"]: (qvec2rotmat(im["qvec"]), np.asarray(im["tvec"], float)) for im in model["images"].values()}


def subset_model(model, keep_names, min_track=2):
    """Keep only images in keep_names; keep 3D points still seen by >= min_track of them.

    gsplat looks every image in the model up in images/ and every track entry up in the model's images,
    so a subset needs both trimmed. Dropped points are set to -1 in the images' observations.
    """
    keep_names = set(keep_names)
    images = {i: dict(im) for i, im in model["images"].items() if im["name"] in keep_names}
    missing = keep_names - {im["name"] for im in images.values()}
    if missing:
        raise KeyError(f"{len(missing)} names not in the model, e.g. {sorted(missing)[:3]}")
    kept_ids = np.array(sorted(images))
    points = {}
    for pid, p in model["points"].items():
        m = np.isin(p["image_ids"], kept_ids)
        if m.sum() >= min_track:
            points[pid] = dict(p, image_ids=p["image_ids"][m], point2D_idxs=p["point2D_idxs"][m])
    for im in images.values():
        ids = np.asarray(im["point3D_ids"], np.int64).copy()
        ids[(ids >= 0) & ~np.isin(ids, np.fromiter(points.keys(), np.int64, len(points)))] = -1
        im["point3D_ids"] = ids
    cams = {c: model["cameras"][c] for c in {im["camera_id"] for im in images.values()}}
    return dict(cameras=cams, images=images, points=points)

In [ ]:
%%writefile /content/work/hybrid_sfm/pose_metrics.py
"""Camera pose accuracy against a reference, numpy only.

Metrics (the usual ones in the VGGT / VGGSfM / MASt3R-SfM papers):
- RRA@τ / RTA@τ: share of image pairs whose relative rotation / relative translation direction error is < τ°.
  Translation direction is sign-ambiguous for some methods, so its error is min(e, 180 - e).
- AUC@T: mean over 30 evenly spaced τ in (0, T] of the share of pairs with max(rotation error, translation error) < τ.
  AUC@30 uses τ = 1..30 (the usual definition). AUC@3 and AUC@5 resolve the sub-degree errors that matter for
  3DGS: in R1, VGGT scored AUC@30 = 0.995 and RRA@5 = 1.0 and still lost 4.2 dB of PSNR.
- focal_error: median relative error of f / image width against the reference (R2).
- ATE: RMSE of camera centres after a Sim(3) (Umeyama) alignment to the reference, divided by the
  reference's scene scale (median distance of reference centres to their mean), in percent.
Images the method failed to register count as 180° error for every pair they are in, so methods that
drop images are not rewarded for it; the share of registered images is reported too.
"""
import numpy as np


def _angle(R):
    return np.degrees(np.arccos(np.clip((np.trace(R) - 1) / 2, -1.0, 1.0)))


def _vec_angle(a, b):
    na, nb = np.linalg.norm(a), np.linalg.norm(b)
    if na < 1e-12 or nb < 1e-12:
        return 180.0
    return np.degrees(np.arccos(np.clip(a @ b / (na * nb), -1.0, 1.0)))


def umeyama(src, dst):
    """s, R, t minimising ||s R src + t - dst||^2 (src, dst: (N, 3))."""
    mu_s, mu_d = src.mean(0), dst.mean(0)
    xs, xd = src - mu_s, dst - mu_d
    U, S, Vt = np.linalg.svd(xd.T @ xs / len(src))
    D = np.eye(3)
    if np.linalg.det(U) * np.linalg.det(Vt) < 0:
        D[2, 2] = -1
    R = U @ D @ Vt
    var_s = (xs ** 2).sum() / len(src)
    s = np.trace(np.diag(S) @ D) / var_s if var_s > 0 else 1.0
    return s, R, mu_d - s * R @ mu_s


def centre(R, t):
    return -R.T @ t


def pose_errors(pred, ref, names=None):
    """pred, ref: {name: (R, t)} world-to-camera. names: the image set to score (default: all of ref)."""
    names = sorted(ref if names is None else names)
    reg = [n for n in names if n in pred]
    out = dict(n_images=len(names), n_registered=len(reg), registered=len(reg) / max(len(names), 1))

    rot_err, tr_err = [], []
    for i in range(len(names)):
        for j in range(i + 1, len(names)):
            a, b = names[i], names[j]
            if a not in pred or b not in pred:
                rot_err.append(180.0); tr_err.append(180.0)
                continue
            (Ra, ta), (Rb, tb) = ref[a], ref[b]
            (Pa, pa), (Pb, pb) = pred[a], pred[b]
            R_ref, R_pred = Rb @ Ra.T, Pb @ Pa.T                       # camera a -> camera b
            t_ref, t_pred = tb - R_ref @ ta, pb - R_pred @ pa
            rot_err.append(_angle(R_pred.T @ R_ref))
            e = _vec_angle(t_pred, t_ref)
            tr_err.append(min(e, 180.0 - e))
    rot_err, tr_err = np.array(rot_err), np.array(tr_err)
    worst = np.maximum(rot_err, tr_err)
    for tau in (1, 5, 15):
        out[f"RRA@{tau}"] = float((rot_err < tau).mean()) if len(worst) else float("nan")
        out[f"RTA@{tau}"] = float((tr_err < tau).mean()) if len(worst) else float("nan")
    for T in (3, 5, 30):
        out[f"AUC@{T}"] = float(np.mean([(worst < t).mean() for t in np.arange(1, 31) * T / 30])) if len(worst) else float("nan")
    out["rot_err_median"] = float(np.median(rot_err)) if len(rot_err) else float("nan")
    out["tr_err_median"] = float(np.median(tr_err)) if len(tr_err) else float("nan")

    if len(reg) >= 3:
        C_ref = np.array([centre(*ref[n]) for n in reg])
        C_pred = np.array([centre(*pred[n]) for n in reg])
        s, R, t = umeyama(C_pred, C_ref)
        res = (s * C_pred @ R.T + t) - C_ref
        scale = np.median(np.linalg.norm(C_ref - C_ref.mean(0), axis=1))
        out["ATE_%"] = float(100 * np.sqrt((res ** 2).sum(1).mean()) / max(scale, 1e-12))
    else:
        out["ATE_%"] = float("nan")
    return out


def _f_over_w(cam):
    p, m = np.asarray(cam["params"], float), cam["model"]
    f = (p[0] + p[1]) / 2 if m in ("PINHOLE", "OPENCV", "OPENCV_FISHEYE", "FULL_OPENCV", "THIN_PRISM_FISHEYE") else p[0]
    return f / cam["width"]


def focal_error(pred_model, ref_model, names=None):
    """Median |f/W - f_ref/W_ref| / (f_ref/W_ref) over registered images (models as in colmap_io)."""
    ref = {im["name"]: _f_over_w(ref_model["cameras"][im["camera_id"]]) for im in ref_model["images"].values()}
    pred = {im["name"]: _f_over_w(pred_model["cameras"][im["camera_id"]]) for im in pred_model["images"].values()}
    names = [n for n in (names if names is not None else ref) if n in pred and n in ref]
    if not names:
        return float("nan")
    return float(np.median([abs(pred[n] - ref[n]) / ref[n] for n in names]))

In [ ]:
%%writefile /content/work/hybrid_sfm/chunks.py
"""R2 chunk sampler: deterministic image subsets of one scene with varied difficulty.

A chunk is what the router decides on. To get labels where each reconstructor sometimes wins and sometimes
fails, chunks vary in size and in baseline:
- window: n consecutive images (images are sorted by name, i.e. capture order): narrow baseline, partial view;
- stride: n images evenly spaced over the whole capture: wide baseline, sparse 360° coverage (R1's bicycle
  setting, where COLMAP registered 11 of 32);
- random: n images drawn at random: irregular overlap.
15 chunks per scene by default (6 window, 5 stride, 4 random); sizes stay <= 40 so one VGGT pass fits a T4 (R1).
"""
import hashlib

import numpy as np

DEFAULT_SPECS = ([("window", 24, k) for k in range(4)] + [("window", 40, k) for k in range(2)]
                 + [("stride", n, 0) for n in (12, 16, 24, 32, 40)]
                 + [("random", n, k) for k, n in enumerate((16, 24, 32, 40))])


def _seed(*parts):
    return int(hashlib.sha256("/".join(map(str, parts)).encode()).hexdigest()[:8], 16)


def make_chunks(scene, names, specs=DEFAULT_SPECS):
    """[{id, scene, kind, n, k, names}] for one scene. Same input -> same chunks, on any machine."""
    names = sorted(names)
    N = len(names)
    out = []
    for kind, n, k in specs:
        n = min(n, N)
        if kind == "window":
            n_windows = sum(1 for s in specs if s[0] == "window" and s[1] == n)
            start = int(round(k * (N - n) / max(n_windows, 1))) if N > n else 0
            idx = np.arange(start, start + n)
        elif kind == "stride":
            idx = np.unique(np.linspace(0, N - 1, n).round().astype(int))
        elif kind == "random":
            idx = np.sort(np.random.default_rng(_seed(scene, kind, n, k)).choice(N, n, replace=False))
        else:
            raise ValueError(kind)
        out.append(dict(id=f"{scene}_{kind}{n}_{k}", scene=scene, kind=kind, n=int(len(idx)), k=k,
                        names=[names[i] for i in idx]))
    return out

In [ ]:
%%writefile /content/work/hybrid_sfm/r2_tools.py
"""R2 helpers on COLMAP models (dicts from colmap_io), numpy only."""
import numpy as np

from . import colmap_io as cio
from . import pose_metrics as pm


def align_to_ref(pred_model, ref_model, names=None):
    """Sim(3) (s, R, t) with ref ≈ s R pred + t, from the camera centres both models share."""
    P, Q = cio.poses_by_name(pred_model), cio.poses_by_name(ref_model)
    common = sorted(n for n in (names if names is not None else P) if n in P and n in Q)
    if len(common) < 3:
        raise ValueError(f"only {len(common)} shared images, need 3 for a Sim(3)")
    C_p = np.array([pm.centre(*P[n]) for n in common])
    C_q = np.array([pm.centre(*Q[n]) for n in common])
    return pm.umeyama(C_p, C_q)


def swap_points(pred_model, ref_model, names=None):
    """The method's cameras with the reference's 3D points moved into the method's frame.

    Controlled initialisation for R2: every method then starts 3DGS from the same point cloud, so a PSNR gap
    comes from poses and intrinsics only (in R1, VGGT started from 100k points and VGGT+BA from 8.9k).
    Tracks are dropped (gsplat only uses them for depth supervision, which R2 doesn't use).
    """
    s, R, t = align_to_ref(pred_model, ref_model, names)
    pts = {}
    for pid, p in ref_model["points"].items():
        x = R.T @ (np.asarray(p["xyz"], float) - t) / s          # ref frame -> method frame
        pts[pid] = dict(xyz=x, rgb=p["rgb"], error=p["error"], image_ids=np.zeros(0, np.int32),
                        point2D_idxs=np.zeros(0, np.int32))
    images = {}
    for iid, im in pred_model["images"].items():
        images[iid] = dict(im, point3D_ids=np.full(len(im["point3D_ids"]), -1, np.int64))
    return dict(cameras=pred_model["cameras"], images=images, points=pts)

In [ ]:
%%writefile /content/work/hybrid_sfm/r2_summary.py
"""Summarise R2 labels (JSON lines written by notebook 05) into r2_summary.md.

One record per (chunk, method, variant, steps):
    chunk, scene, kind, n, method ('ref' | 'classical' | 'vggt' | 'vggt_ba'), variant ('own' | 'refpts'), steps,
    status ('ok' | 'recon_failed' | 'low_coverage' | 'train_failed'), recon {wall_s, ...}, pose {...} | None,
    focal_err, gs {psnr, ssim, lpips, num_GS, train_s} | None
Later records replace earlier ones with the same key, so several runs' files can simply be concatenated.

Routing loss of method m on chunk c: L = PSNR_ref(c) - PSNR_m(c) (dB below the official poses), or FAIL_LOSS if m
produced no usable twin. Usage:
    python -m hybrid_sfm.r2_summary labels.jsonl [more.jsonl ...] [--out r2_summary.md] [--steps 3000 --long 7000]
"""
import argparse
import json
from collections import defaultdict

import numpy as np

METHODS = ["classical", "vggt", "vggt_ba"]
FAIL_LOSS = 10.0
POSE_KEYS = ["AUC@30", "AUC@5", "AUC@3", "RRA@5", "rot_err_median", "ATE_%", "focal_err"]


def load(paths):
    recs = {}
    for p in paths:
        with open(p, encoding="utf-8") as f:
            for line in f:
                if line.strip():
                    r = json.loads(line)
                    recs[(r["chunk"], r["method"], r.get("variant", "own"), r["steps"])] = r
    return list(recs.values())


def rankdata(x):
    x = np.asarray(x, float)
    order = np.argsort(x, kind="mergesort")
    ranks = np.empty(len(x))
    ranks[order] = np.arange(len(x))
    for v in np.unique(x):                      # average ranks for ties
        m = x == v
        ranks[m] = ranks[m].mean()
    return ranks


def spearman(a, b):
    a, b = np.asarray(a, float), np.asarray(b, float)
    ok = np.isfinite(a) & np.isfinite(b)
    if ok.sum() < 3:
        return float("nan"), int(ok.sum())
    ra, rb = rankdata(a[ok]), rankdata(b[ok])
    if ra.std() == 0 or rb.std() == 0:
        return float("nan"), int(ok.sum())
    return float(np.corrcoef(ra, rb)[0, 1]), int(ok.sum())


def psnr(r):
    return r["gs"]["psnr"] if r and r.get("status") == "ok" and r.get("gs") else None


def summarize(records, steps=3000, long_steps=7000, fail_loss=FAIL_LOSS):
    idx = {(r["chunk"], r["method"], r.get("variant", "own"), r["steps"]): r for r in records}
    chunks = sorted({r["chunk"] for r in records})
    meta = {r["chunk"]: r for r in records}
    get = lambda c, m, v="own", s=steps: idx.get((c, m, v, s))
    L = ["# R2 label summary", ""]
    res = dict(n_chunks=len(chunks))

    # ---- 1. what ran
    L += [f"**{len(chunks)} chunks** from {len({meta[c]['scene'] for c in chunks})} scenes; 3DGS labels at {steps} steps.", "",
          "| Method | Reconstructed | ≥ 90% registered | 3DGS trained | Mean pose time |", "|---|---|---|---|---|"]
    for m in ["ref"] + METHODS:
        rs = [get(c, m) for c in chunks]
        rs = [r for r in rs if r]
        rec_ok = [r for r in rs if r["status"] != "recon_failed"]
        cov = [r for r in rec_ok if r["status"] != "low_coverage"]
        tr = [r for r in rs if r["status"] == "ok"]
        t = [r["recon"].get("wall_s", np.nan) for r in rec_ok if r.get("recon")]
        L.append(f"| {m} | {len(rec_ok)}/{len(rs)} | {len(cov)}/{len(rs)} | {len(tr)}/{len(rs)} | "
                 + ("–" if m == "ref" or not t else f"{np.nanmean(t):.0f} s") + " |")

    # ---- 2. losses and routing baselines
    loss, cost = defaultdict(dict), defaultdict(dict)
    for c in chunks:
        p_ref = psnr(get(c, "ref"))
        if p_ref is None:
            continue
        for m in METHODS:
            r = get(c, m)
            if r is None:
                continue
            p = psnr(r)
            loss[c][m] = fail_loss if p is None else p_ref - p
            cost[c][m] = (r.get("recon") or {}).get("wall_s", np.nan)
    full = [c for c in loss if all(m in loss[c] for m in METHODS)]
    res["n_labelled"] = len(full)
    L += ["", f"**Routing on {len(full)} chunks with all labels** (loss = dB below the official poses; a method "
          f"with no usable twin costs {fail_loss:.0f} dB):", "",
          "| Policy | Mean loss | Median loss | Failures | Mean pose time | Best on |", "|---|---|---|---|---|---|"]
    wins = {m: sum(1 for c in full if min(loss[c], key=loss[c].get) == m) for m in METHODS}
    pol = {}
    for m in METHODS:
        l = [loss[c][m] for c in full]
        pol[f"always {m}"] = dict(mean=float(np.mean(l)) if l else np.nan, median=float(np.median(l)) if l else np.nan,
                                  fails=sum(x >= fail_loss for x in l), time=float(np.nanmean([cost[c][m] for c in full])) if full else np.nan,
                                  wins=wins[m])
    lo = [min(loss[c].values()) for c in full]
    pol["oracle (best per chunk)"] = dict(mean=float(np.mean(lo)) if lo else np.nan, median=float(np.median(lo)) if lo else np.nan,
                                          fails=sum(x >= fail_loss for x in lo),
                                          time=float(np.nanmean([cost[c][min(loss[c], key=loss[c].get)] for c in full])) if full else np.nan,
                                          wins=None)
    for k, v in pol.items():
        L.append(f"| {k} | {v['mean']:.2f} dB | {v['median']:.2f} dB | {v['fails']} | {v['time']:.0f} s | "
                 + ("–" if v["wins"] is None else f"{v['wins']}/{len(full)}") + " |")
    res["policies"] = pol

    # ---- 3. by chunk kind
    kinds = sorted({meta[c]["kind"] for c in full})
    L += ["", "**Mean loss by chunk kind** (window = consecutive images, stride = spread over the capture, random):", "",
          "| Kind | Chunks | " + " | ".join(METHODS) + " | Oracle |", "|---|---|" + "---|" * (len(METHODS) + 1)]
    for k in kinds:
        cs = [c for c in full if meta[c]["kind"] == k]
        L.append(f"| {k} | {len(cs)} | " + " | ".join(f"{np.mean([loss[c][m] for c in cs]):.2f}" for m in METHODS)
                 + f" | {np.mean([min(loss[c].values()) for c in cs]):.2f} |")

    # ---- 4. H1 preview: do pose metrics predict the twin's loss?
    rows = [(c, m) for c in loss for m in loss[c] if loss[c][m] < fail_loss and (get(c, m) or {}).get("pose")]
    L += ["", f"**Do pose metrics predict the twin's loss?** Spearman ρ over {len(rows)} trained (chunk, method) pairs. "
          "A perfect predictor has |ρ| = 1; for accuracy metrics (AUC, RRA) the expected sign is negative:", "",
          "| Pose metric | ρ with loss | n |", "|---|---|---|"]
    res["pose_vs_loss"] = {}
    for k in POSE_KEYS:
        vals = [(get(c, m)["focal_err"] if k == "focal_err" else get(c, m)["pose"].get(k)) for c, m in rows]
        rho, n = spearman([np.nan if v is None else v for v in vals], [loss[c][m] for c, m in rows])
        res["pose_vs_loss"][k] = rho
        L.append(f"| {k} | {rho:+.2f} | {n} |")

    # ---- 5. fidelity: short vs long recipe
    pairs = [(psnr(get(c, m, "own", steps)), psnr(get(c, m, "own", long_steps)))
             for c in chunks for m in ["ref"] + METHODS]
    pairs = [(a, b) for a, b in pairs if a is not None and b is not None]
    rho, n = spearman([a for a, _ in pairs], [b for _, b in pairs])
    same_best, n_cmp = 0, 0
    for c in chunks:
        a = {m: psnr(get(c, m, "own", steps)) for m in METHODS}
        b = {m: psnr(get(c, m, "own", long_steps)) for m in METHODS}
        if all(v is not None for v in list(a.values()) + list(b.values())):
            n_cmp += 1
            same_best += max(a, key=a.get) == max(b, key=b.get)
    res["fidelity"] = dict(rho=rho, n=n, same_best=same_best, n_chunks=n_cmp)
    L += ["", f"**Label fidelity ({steps} vs {long_steps} steps):** Spearman ρ = {rho:.2f} over {n} runs; "
          f"same best method on {same_best}/{n_cmp} chunks."]

    # ---- 6. initialisation control
    L += ["", "**Initialisation control** (same chunks, each method trained from its own points vs from the reference "
          "points moved into its frame; mean loss in dB):", "", "| Method | Chunks | Own points | Reference points | Difference |",
          "|---|---|---|---|---|"]
    res["init_control"] = {}
    for m in METHODS:
        cs = [c for c in loss if psnr(get(c, m, "refpts")) is not None and psnr(get(c, m)) is not None]
        if not cs:
            continue
        own = np.mean([psnr(get(c, "ref")) - psnr(get(c, m)) for c in cs])
        ref = np.mean([psnr(get(c, "ref")) - psnr(get(c, m, "refpts")) for c in cs])
        res["init_control"][m] = dict(n=len(cs), own=float(own), refpts=float(ref))
        L.append(f"| {m} | {len(cs)} | {own:.2f} | {ref:.2f} | {own - ref:+.2f} |")
    return "\n".join(L) + "\n", res


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("labels", nargs="+")
    ap.add_argument("--out", default="r2_summary.md")
    ap.add_argument("--steps", type=int, default=3000)
    ap.add_argument("--long", type=int, default=7000)
    a = ap.parse_args()
    md, res = summarize(load(a.labels), a.steps, a.long)
    open(a.out, "w", encoding="utf-8").write(md)
    json.dump(res, open(a.out.replace(".md", ".json"), "w"), indent=1, default=float)
    print(md)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/work/hybrid_sfm/run_classical.py
"""R2 classical reconstructor: ALIKED + LightGlue matches (GPU) -> COLMAP verification and incremental mapping.

R1's COLMAP baseline (SIFT, exhaustive CPU matching) took 16-19 min for 32 images, far too slow for 100+ chunks.
This is the hloc-style pipeline from the proposal: learned features and matcher on the GPU, then COLMAP's own
geometric verification (pycolmap.verify_matches) and incremental mapper. One shared PINHOLE camera, as in R1.
Falls back to R1's SIFT pipeline if LightGlue can't be loaded (recorded in classical_info.json).

Usage (VGGT environment: pycolmap 3.10, torch, LightGlue):
    python run_classical.py <scene_dir> [--max_kpts 2048]      # reads <scene_dir>/images, writes <scene_dir>/sparse/0
"""
import argparse
import itertools
import json
import os
import shutil
import sqlite3
import time

import numpy as np
import pycolmap

MAX_IMAGE_ID = 2147483647          # COLMAP's pair-id encoding


def pair_id(i, j):
    return i * MAX_IMAGE_ID + j if i < j else j * MAX_IMAGE_ID + i


def extract_and_match(image_dir, names, max_kpts=2048):
    """ALIKED keypoints at native resolution, LightGlue matches for every pair. Returns (kpts, matches)."""
    import torch
    from lightglue import ALIKED, LightGlue
    from lightglue.utils import load_image, rbd
    dev = "cuda" if torch.cuda.is_available() else "cpu"
    extractor = ALIKED(max_num_keypoints=max_kpts).eval().to(dev)
    matcher = LightGlue(features="aliked").eval().to(dev)
    feats, kpts, matches = {}, {}, {}
    with torch.inference_mode():
        for n in names:
            feats[n] = extractor.extract(load_image(os.path.join(image_dir, n)).to(dev), resize=None)
            kpts[n] = rbd(feats[n])["keypoints"].cpu().numpy().astype(np.float32)
        for a, b in itertools.combinations(names, 2):
            out = rbd(matcher({"image0": feats[a], "image1": feats[b]}))
            matches[(a, b)] = out["matches"].cpu().numpy().astype(np.uint32)
    return kpts, matches


_SCHEMA = [   # COLMAP 3.x tables (as in hloc / COLMAP's database.py); only used if pycolmap can't create the file
    "CREATE TABLE IF NOT EXISTS cameras (camera_id INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL, model INTEGER NOT NULL, "
    "width INTEGER NOT NULL, height INTEGER NOT NULL, params BLOB, prior_focal_length INTEGER NOT NULL)",
    "CREATE TABLE IF NOT EXISTS images (image_id INTEGER PRIMARY KEY AUTOINCREMENT NOT NULL, name TEXT NOT NULL UNIQUE, "
    "camera_id INTEGER NOT NULL, CONSTRAINT image_id_check CHECK(image_id >= 0 and image_id < 2147483647), "
    "FOREIGN KEY(camera_id) REFERENCES cameras(camera_id))",
    "CREATE TABLE IF NOT EXISTS keypoints (image_id INTEGER PRIMARY KEY NOT NULL, rows INTEGER NOT NULL, "
    "cols INTEGER NOT NULL, data BLOB, FOREIGN KEY(image_id) REFERENCES images(image_id) ON DELETE CASCADE)",
    "CREATE TABLE IF NOT EXISTS descriptors (image_id INTEGER PRIMARY KEY NOT NULL, rows INTEGER NOT NULL, "
    "cols INTEGER NOT NULL, data BLOB, FOREIGN KEY(image_id) REFERENCES images(image_id) ON DELETE CASCADE)",
    "CREATE TABLE IF NOT EXISTS matches (pair_id INTEGER PRIMARY KEY NOT NULL, rows INTEGER NOT NULL, "
    "cols INTEGER NOT NULL, data BLOB)",
    "CREATE TABLE IF NOT EXISTS two_view_geometries (pair_id INTEGER PRIMARY KEY NOT NULL, rows INTEGER NOT NULL, "
    "cols INTEGER NOT NULL, data BLOB, config INTEGER NOT NULL, F BLOB, E BLOB, H BLOB, qvec BLOB, tvec BLOB)",
    "CREATE UNIQUE INDEX IF NOT EXISTS index_name ON images(name)",
]


def create_db(db_path):
    """An empty COLMAP database in this pycolmap's own schema (4.x adds rig/frame tables)."""
    for make in (lambda: pycolmap.Database.open(db_path), lambda: pycolmap.Database(db_path)):
        try:
            db = make()
            if hasattr(db, "close"):
                db.close()
            if os.path.exists(db_path):
                return "pycolmap"
        except (TypeError, AttributeError, ValueError, RuntimeError):
            pass
    con = sqlite3.connect(db_path)
    for q in _SCHEMA:
        con.execute(q)
    con.commit()
    con.close()
    return "sql"


def import_images(db_path, image_dir, names):
    """Cameras + images rows: one shared PINHOLE camera (focal prior 1.2 x max side, refined by BA)."""
    create_db(db_path)
    ro = pycolmap.ImageReaderOptions()
    ro.camera_model = "PINHOLE"
    pycolmap.import_images(db_path, image_dir, pycolmap.CameraMode.SINGLE, names, ro)   # positional: 3.10 and 4.x name args differently


def write_features(db_path, kpts, matches, min_matches=15):
    """Keypoints and raw matches into a database made by import_images (COLMAP's pixel-centre convention)."""
    con = sqlite3.connect(db_path)
    ids = {name: iid for iid, name in con.execute("SELECT image_id, name FROM images")}
    for n, k in kpts.items():
        k = np.ascontiguousarray(k + 0.5, np.float32)          # (0, 0) = centre of the top-left pixel -> COLMAP's corner origin
        con.execute("INSERT INTO keypoints(image_id, rows, cols, data) VALUES (?, ?, ?, ?)",
                    (ids[n], k.shape[0], 2, k.tobytes()))
    pairs = []
    for (a, b), m in matches.items():
        if len(m) < min_matches:
            continue
        m = np.asarray(m, np.uint32).reshape(-1, 2)
        if ids[a] > ids[b]:
            m = m[:, ::-1]
        m = np.ascontiguousarray(m)
        con.execute("INSERT INTO matches(pair_id, rows, cols, data) VALUES (?, ?, ?, ?)",
                    (pair_id(ids[a], ids[b]), m.shape[0], 2, m.tobytes()))
        pairs.append((a, b))
    con.commit()
    con.close()
    return pairs


def verify_and_map(db_path, image_dir, pairs, work):
    pairs_txt = os.path.join(work, "pairs.txt")
    with open(pairs_txt, "w") as f:
        f.write("\n".join(f"{a} {b}" for a, b in pairs) + "\n")
    pycolmap.verify_matches(db_path, pairs_txt)
    return pycolmap.incremental_mapping(db_path, image_dir, os.path.join(work, "mapping"))


def sift_pipeline(db_path, image_dir, names, work):
    """R1's COLMAP baseline (fallback)."""
    ro = pycolmap.ImageReaderOptions()
    ro.camera_model = "PINHOLE"
    try:
        pycolmap.extract_features(db_path, image_dir, image_list=names, camera_mode=pycolmap.CameraMode.SINGLE, reader_options=ro)
    except TypeError:
        pycolmap.extract_features(db_path, image_dir, camera_mode=pycolmap.CameraMode.SINGLE, reader_options=ro)
    pycolmap.match_exhaustive(db_path)
    return pycolmap.incremental_mapping(db_path, image_dir, os.path.join(work, "mapping"))


def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("scene")
    ap.add_argument("--max_kpts", type=int, default=2048)
    ap.add_argument("--sift", action="store_true", help="skip LightGlue, use R1's SIFT pipeline")
    a = ap.parse_args()
    image_dir, work = os.path.join(a.scene, "images"), os.path.join(a.scene, "classical_work")
    shutil.rmtree(work, ignore_errors=True)
    os.makedirs(work)
    db = os.path.join(work, "database.db")
    names = sorted(os.listdir(image_dir))
    info = dict(n_images=len(names), pycolmap=pycolmap.__version__)
    t0 = time.time()
    kpts = None
    if not a.sift:
        try:
            kpts, matches = extract_and_match(image_dir, names, a.max_kpts)
        except Exception as e:          # no LightGlue, weights not downloadable, ...: R1's SIFT pipeline instead
            print("ALIKED + LightGlue unavailable, using SIFT:", repr(e)[:300])
            info["fallback_reason"] = repr(e)[:300]
    if kpts is not None:
        info.update(features="aliked+lightglue", match_s=round(time.time() - t0, 1),
                    mean_kpts=float(np.mean([len(k) for k in kpts.values()])))
        t1 = time.time()
        import_images(db, image_dir, names)
        pairs = write_features(db, kpts, matches)
        info["pairs_with_matches"] = len(pairs)
        recs = verify_and_map(db, image_dir, pairs, work)
    else:
        t1 = time.time()
        info.update(features="sift", match_s=0.0)
        create_db(db)
        recs = sift_pipeline(db, image_dir, names, work)
    info["map_s"] = round(time.time() - t1, 1)
    info["total_s"] = round(time.time() - t0, 1)
    if not recs:
        json.dump(dict(info, registered=0), open(os.path.join(a.scene, "classical_info.json"), "w"), indent=1)
        raise SystemExit("classical: no reconstruction")
    best = max(recs.values(), key=lambda r: r.num_reg_images())
    out = os.path.join(a.scene, "sparse", "0")
    os.makedirs(out, exist_ok=True)
    best.write(out)
    info.update(models=len(recs), registered=best.num_reg_images(), points=best.num_points3D())
    json.dump(info, open(os.path.join(a.scene, "classical_info.json"), "w"), indent=1)
    print(info)


if __name__ == "__main__":
    main()

In [ ]:
%%writefile /content/work/hybrid_sfm/vggt_colmap_lowmem.py
"""VGGT's demo_colmap.py with less GPU memory, plus an optional dump of VGGT's raw predictions.

demo_colmap.py keeps the 1B-parameter model (~5 GB in fp32) on the GPU while it predicts tracks for bundle
adjustment, which runs out of memory at 32 images on a T4. This wrapper moves the model to the CPU as soon as
the forward pass is done. Everything else is VGGT's own code and arguments.
Extra flags:
    --no_fine_tracking   demo_colmap's --fine_tracking defaults to True and can't be turned off there
    --save_pred X.npz    save VGGT's cameras, depth and depth confidence (1/4 resolution, fp16) for R3's signals
Run from the vggt repo folder (or with it on PYTHONPATH):
    python vggt_colmap_lowmem.py --scene_dir X [--use_ba] [--max_query_pts 2048 --query_frame_num 5] [--no_fine_tracking]
"""
import glob
import os
import sys

import numpy as np
import torch

import demo_colmap as dc

_run_vggt = dc.run_VGGT
SAVE_PRED, SCENE_DIR = None, None


def _np(x):
    return x.detach().float().cpu().numpy() if torch.is_tensor(x) else np.asarray(x, dtype=np.float32)


def save_pred(out, path, scene_dir):
    extrinsic, intrinsic, depth, conf = (_np(x) for x in out[:4])
    depth, conf = depth.reshape(depth.shape[0], depth.shape[1], depth.shape[2]), conf.reshape(conf.shape[0], conf.shape[1], conf.shape[2])
    names = [os.path.basename(p) for p in glob.glob(os.path.join(scene_dir, "images", "*"))]   # demo_colmap's own order
    np.savez_compressed(path, names=np.array(names), extrinsic=extrinsic.astype(np.float32),
                        intrinsic=intrinsic.astype(np.float32), depth=depth[:, ::4, ::4].astype(np.float16),
                        conf=conf[:, ::4, ::4].astype(np.float16), resolution=np.array(depth.shape[1:3]))
    print(f"saved VGGT predictions for {len(names)} images to {path}")


def _run_then_free(model, *args, **kwargs):
    out = _run_vggt(model, *args, **kwargs)
    model.cpu()
    torch.cuda.empty_cache()
    if SAVE_PRED:
        try:
            save_pred(out, SAVE_PRED, SCENE_DIR)
        except Exception as e:          # never lose the reconstruction over the extra file
            print("could not save VGGT predictions:", repr(e))
    return out


dc.run_VGGT = _run_then_free

if __name__ == "__main__":
    no_fine = "--no_fine_tracking" in sys.argv
    if no_fine:
        sys.argv.remove("--no_fine_tracking")
    if "--save_pred" in sys.argv:
        i = sys.argv.index("--save_pred")
        SAVE_PRED = sys.argv[i + 1]
        del sys.argv[i:i + 2]
    args = dc.parse_args()
    SCENE_DIR = args.scene_dir
    if no_fine:
        args.fine_tracking = False
    with torch.no_grad():
        dc.demo_fn(args)

In [ ]:
open(f'{WORK}/hybrid_sfm/__init__.py', 'a').close()
sys.path.insert(0, WORK)
from hybrid_sfm import colmap_io as cio, pose_metrics as pm, chunks, r2_tools
print('hybrid_sfm ready')

## 5. Data: Mip-NeRF 360 scenes (one resolution + official poses), downloaded one at a time

In [ ]:
from remotezip import RemoteZip
import imageio.v2 as imageio, numpy as np
URL = 'http://storage.googleapis.com/gresearch/refraw360/360_v2.zip'

def get_scene(scene):
    # Downloads only the files that are missing or incomplete, so an interrupted download is finished next time.
    data, want = f'{WORK}/data/360_v2/{scene}', ('sparse/', 'poses_bounds.npy', RES[scene] + '/')
    if not os.path.exists(f'{data}/.complete_{RES[scene]}'):
        t = time.time(); n = 0
        with RemoteZip(URL) as z:
            for info in z.infolist():
                m = info.filename
                if not (m.startswith(f'{scene}/') and not m.endswith('/') and m.split('/', 1)[1].startswith(want)):
                    continue
                dst = f'{data}/' + m.split('/', 1)[1]
                if os.path.exists(dst) and os.path.getsize(dst) == info.file_size:
                    continue
                os.makedirs(os.path.dirname(dst), exist_ok=True)
                with z.open(m) as src_, open(dst + '.part', 'wb') as f:
                    f.write(src_.read())
                os.replace(dst + '.part', dst); n += 1
        open(f'{data}/.complete_{RES[scene]}', 'w').write('ok')
        print(f'{scene}: downloaded {n} files in {time.time()-t:.0f}s')
    return data

## 6. Chunks (deterministic: the same on every machine and every run)

In [ ]:
SCENE_REF, CHUNKS = {}, []
for scene in SCENES:
    data = get_scene(scene)
    ref = cio.read_model(data)                                     # official model (all images)
    in_model = {im['name'] for im in ref['images'].values()}
    names = [n for n in os.listdir(f'{data}/{RES[scene]}') if n in in_model]
    SCENE_REF[scene] = dict(data=data, ref=ref, poses=cio.poses_by_name(ref))
    cs = chunks.make_chunks(scene, names)
    CHUNKS += cs
    w = imageio.imread(f'{data}/{RES[scene]}/{names[0]}').shape
    print(f'{scene:>8}: {len(names)} images at {w[1]}x{w[0]} -> {len(cs)} chunks')
json.dump(CHUNKS, open(f'{OUT}/r2_chunks.json', 'w'), indent=0)

def load_done():
    done = {}
    if os.path.exists(LABELS):
        for line in open(LABELS):
            if line.strip():
                r = json.loads(line); done[(r['chunk'], r['method'], r['variant'], r['steps'])] = r
    return done
DONE = load_done()
todo = [c for c in CHUNKS if not all((c['id'], m, 'own', STEPS) in DONE for m in METHODS)]
print(f'{len(CHUNKS)} chunks, {len(CHUNKS) - len(todo)} already labelled, {len(todo)} to go')

## 7. Reconstruct, score, train

For every chunk and method:
1. reconstruct (with retries and a 30 min timeout);
2. score the poses against the official ones;
3. if ≥ 90% of the chunk is registered, train the short 3DGS recipe;
4. append one line to `r2_labels.jsonl`.

Any failure becomes a label too: failures are exactly what the router has to predict.

In [ ]:
MIN_POINTS = dict(vggt=1000, vggt_ba=50, classical=50)   # gsplat starts from the model's points (F-R1.2)
GS_ENV = {**os.environ, 'PYTORCH_CUDA_ALLOC_CONF': 'expandable_segments:True'}
help_txt = subprocess.run([sys.executable, 'simple_trainer.py', 'mcmc', '--help'], capture_output=True, text=True,
                          cwd=f'{REPO}/examples').stdout
VIDEO_FLAG = ['--disable_video'] if re.search(r'disable.video', help_txt) else []      # F-R1.4

def append_label(rec):
    with open(LABELS, 'a') as f:
        f.write(json.dumps(rec, default=float) + '\n')
    DONE[(rec['chunk'], rec['method'], rec['variant'], rec['steps'])] = rec

def save_log(name, text):
    open(f'{OUT}/r2_logs/{name}.txt', 'w').write(text[-20000:])

def prepare(c):
    # One folder per method with the chunk's images (symlinks); 'ref' gets the official model cut to the chunk.
    sr = SCENE_REF[c['scene']]
    for m in METHODS:
        d = f"{R2}/{c['id']}/{m}"
        os.makedirs(f'{d}/images', exist_ok=True)
        for n in c['names']:
            if not os.path.exists(f'{d}/images/{n}'):
                os.symlink(f"{sr['data']}/{RES[c['scene']]}/{n}", f'{d}/images/{n}')
    d = f"{R2}/{c['id']}/ref"
    if not os.path.exists(f'{d}/sparse/0/images.bin'):
        cio.write_model_bin(cio.subset_model(sr['ref'], c['names']), f'{d}/sparse/0')

def model_path(c, m):
    d = f"{R2}/{c['id']}/{m}"
    return f'{d}/sparse/0' if os.path.exists(f'{d}/sparse/0/images.bin') else f'{d}/sparse'

def reconstruct(c, m):
    # Runs one reconstructor on one chunk; returns info (status, wall_s, points, ...). Model saved to OUT.
    d, cid = f"{R2}/{c['id']}/{m}", c['id']
    saved = f'{OUT}/r2_models/{cid}/{m}'
    if os.path.exists(f'{saved}/info.json'):                    # done in an earlier session
        info = json.load(open(f'{saved}/info.json'))
        if info['status'] == 'ok' and not os.path.exists(f'{d}/sparse'):
            shutil.copytree(f'{saved}/sparse', f'{d}/sparse')
        return info
    lowmem = [VPY, f'{WORK}/hybrid_sfm/vggt_colmap_lowmem.py', '--scene_dir', d]
    if m == 'classical':
        tries = [([VPY, f'{WORK}/hybrid_sfm/run_classical.py', d], WORK)]
    elif m == 'vggt':
        pred = ['--save_pred', f'{OUT}/r2_pred/{cid}.npz']
        tries = [(lowmem + ['--conf_thres_value', '2'] + pred, VGGT_REPO), (lowmem + ['--conf_thres_value', '1.01'] + pred, VGGT_REPO)]
    else:
        tries = [(lowmem + ['--use_ba', '--max_query_pts', '2048', '--query_frame_num', '5', '--no_fine_tracking'], VGGT_REPO)]
    for k, (cmd, cwd) in enumerate(tries):
        shutil.rmtree(f'{d}/sparse', ignore_errors=True)
        t = time.time()
        try:
            r = subprocess.run(cmd, cwd=cwd, env=VENV_ENV, capture_output=True, text=True, timeout=1800)
            rc, log = r.returncode, r.stdout + '\n----- stderr -----\n' + r.stderr
        except subprocess.TimeoutExpired as e:
            rc, log = -1, f'timeout after 1800 s\n{e.stdout or ""}\n{e.stderr or ""}'
        info = dict(method=m, wall_s=round(time.time() - t, 1), args=' '.join(cmd[2:]).replace(d, '<chunk>'), attempt=k)
        if rc != 0:
            info['status'] = 'failed: ' + ([l for l in log.strip().splitlines() if l.strip()] or ['?'])[-1][:200]
            save_log(f'{cid}_{m}_{k}', log); continue
        try:
            info['points'] = len(cio.read_model(d)['points'])
        except Exception as e:
            info['points'] = 0
        if m == 'classical' and os.path.exists(f'{d}/classical_info.json'):
            info['classical'] = json.load(open(f'{d}/classical_info.json'))
        if info['points'] < MIN_POINTS[m]:
            info['status'] = f"failed: only {info['points']} 3D points"; save_log(f'{cid}_{m}_{k}', log); continue
        info['status'] = 'ok'
        break
    os.makedirs(saved, exist_ok=True)
    if info['status'] == 'ok':
        shutil.copytree(f'{d}/sparse', f'{saved}/sparse', dirs_exist_ok=True)
    json.dump(info, open(f'{saved}/info.json', 'w'), indent=1)
    return info

def train_gs(data_dir, steps, tag):
    # Same short recipe for every method; returns val + train stats. The run folder is deleted afterwards.
    rd = f'{WORK}/gs_runs/{tag}'
    shutil.rmtree(rd, ignore_errors=True)
    cmd = [sys.executable, 'simple_trainer.py', 'mcmc', '--strategy.cap-max', str(CAP), '--data_dir', data_dir,
           '--data_factor', '1', '--result_dir', rd, '--max_steps', str(steps), '--eval_steps', str(steps),
           '--save_steps', str(steps), '--disable_viewer', '--lpips_net', 'vgg', *VIDEO_FLAG]
    t = time.time()
    r = subprocess.run(cmd, cwd=f'{REPO}/examples', capture_output=True, text=True, env=GS_ENV, timeout=3600)
    vals = sorted(glob.glob(f'{rd}/stats/val_step*.json'))
    if r.returncode != 0 or not vals:
        save_log(f'train_{tag}', r.stdout + '\n----- stderr -----\n' + r.stderr)
        raise RuntimeError('3DGS failed: ' + ([l for l in r.stderr.strip().splitlines() if l.strip()] or ['?'])[-1][:200])
    v = json.load(open(vals[-1]))
    tr = sorted(glob.glob(f'{rd}/stats/train_step*.json'))
    mem = json.load(open(tr[-1])).get('mem') if tr else None
    shutil.rmtree(rd, ignore_errors=True)
    return dict(psnr=v['psnr'], ssim=v['ssim'], lpips=v['lpips'], num_GS=v.get('num_GS'), mem_gb=mem,
                train_s=round(time.time() - t, 1))

def score(c, m):
    # Pose metrics and focal error against the official poses, on the chunk's images.
    sr = SCENE_REF[c['scene']]
    model = cio.read_model(model_path(c, m))
    pose = pm.pose_errors(cio.poses_by_name(model), sr['poses'], c['names'])
    return model, pose, pm.focal_error(model, sr['ref'], c['names'])

def label(c, m, variant, steps, info, pose, focal, data_dir):
    rec = dict(chunk=c['id'], scene=c['scene'], kind=c['kind'], n=c['n'], method=m, variant=variant, steps=steps,
               recon=info, pose=pose, focal_err=focal, gs=None)
    if info.get('status') != 'ok':
        rec['status'] = 'recon_failed'
    elif pose is not None and pose['registered'] < MIN_COVERAGE:
        rec['status'] = 'low_coverage'
    else:
        try:
            rec['gs'] = train_gs(data_dir, steps, f"{c['id']}_{m}_{variant}_{steps}"); rec['status'] = 'ok'
        except Exception as e:
            rec['status'] = 'train_failed'; rec['error'] = str(e)[:300]
    append_label(rec)
    return rec

def run_chunk(c):
    prepare(c)
    fid = any(c['id'].endswith('_' + f) for f in FIDELITY)
    for m in METHODS:
        need = [(v, s) for v, s in [('own', STEPS)] + ([('own', LONG_STEPS)] + ([('refpts', STEPS)] if m != 'ref' else []) if fid else [])
                if (c['id'], m, v, s) not in DONE]
        if not need:
            continue
        info = dict(method='ref', status='ok', wall_s=0.0) if m == 'ref' else reconstruct(c, m)
        pose = focal = model = None
        if info['status'] == 'ok':
            model, pose, focal = score(c, m)
        for variant, steps in need:
            data_dir = f"{R2}/{c['id']}/{m}"
            if variant == 'refpts' and model is not None and pose['registered'] >= MIN_COVERAGE:
                data_dir = f'{data_dir}_refpts'
                os.makedirs(data_dir, exist_ok=True)
                if not os.path.exists(f'{data_dir}/images'):
                    os.symlink(f"{R2}/{c['id']}/{m}/images", f'{data_dir}/images')
                ref_sub = cio.read_model(model_path(c, 'ref'))
                cio.write_model_bin(r2_tools.swap_points(model, ref_sub, c['names']), f'{data_dir}/sparse/0')
            rec = label(c, m, variant, steps, info, pose, focal, data_dir)
            g = rec['gs']
            print(f"  {m:<9} {variant:<6} {steps:>5}: {rec['status']:<13}"
                  + (f" reg {pose['n_registered']}/{pose['n_images']} AUC@3 {pose['AUC@3']:.2f} AUC@30 {pose['AUC@30']:.2f}" if pose else '')
                  + (f" | PSNR {g['psnr']:.2f} ({g['train_s']:.0f} s)" if g else ''))
    shutil.rmtree(f"{R2}/{c['id']}", ignore_errors=True)   # local copies only; models and labels are in OUT

## 8. Run (resumable: labelled runs are skipped)

In [ ]:
for i, c in enumerate(todo):
    if (time.time() - T_START) / 3600 > BUDGET_H:
        print(f'⏸ time budget of {BUDGET_H} h reached: {len(todo) - i} chunks left for the next run'); break
    t = time.time()
    print(f"[{i + 1}/{len(todo)}] {c['id']} ({c['n']} images, {c['kind']})")
    try:
        run_chunk(c)
    except Exception as e:                                       # one bad chunk must not stop the run
        print(f"  ❌ {c['id']}: {repr(e)[:300]}")
    print(f'  {(time.time() - t) / 60:.1f} min | {(time.time() - T_START) / 3600:.2f} h elapsed')

## 9. Summary (`r2/r2_summary.md`: paste it into the chat)

In [ ]:
from hybrid_sfm import r2_summary
md_, res_ = r2_summary.summarize(r2_summary.load([LABELS]), STEPS, LONG_STEPS)
open(f'{OUT}/r2_summary.md', 'w', encoding='utf-8').write(md_)
json.dump(res_, open(f'{OUT}/r2_summary.json', 'w'), indent=1, default=float)
print(md_)
print(subprocess.run(['du', '-sh', OUT], capture_output=True, text=True).stdout)

## ✅ Checklist
- [ ] Paste `r2_summary.md` into the chat
- [ ] If some scenes are left (time budget), run again (Kaggle: add this run's Output as an Input)
- [ ] Keep `r2_labels.jsonl`, `r2_pred/` and `r2_models/`: they are R3's training data